# Qwen3-ASR Generic Colab Workflow

Self-contained Colab workflow for Qwen3-ASR on arbitrary audio inputs.

Supported inputs:

- Existing Google Drive/local path
- Direct HTTP(S) audio/media URL
- Xiaoyuzhou episode URL, with built-in preset manifest

Pipeline:

1. Mount Drive and install repo/helpers.
2. Choose runtime preset: T4 / A100 / H100.
3. Resolve/download audio.
4. Normalize to 16 kHz mono WAV.
5. Chunk and transcribe with checkpoint/resume.
6. Save JSONL/CSV/Markdown/Text to Drive.
7. Optional Colab GenAI text-only evaluation.
8. Optional high-fidelity proofread.
9. Optional enhancements: highlights, translation, mindmap, research leads.

Important honesty note: Colab GenAI/proofreading is text-only. It cannot listen to audio and cannot verify ASR faithfulness.

Benign warning note: `Setting pad_token_id to eos_token_id:151645 for open-end generation` is a normal Transformers generation warning for decoder-only models. It is not an ASR error.


In [ ]:
#@title 0. Mount Drive and configure durable paths
from pathlib import Path
import os, sys, json, subprocess, time, shutil

MOUNT_DRIVE = True  #@param {type:"boolean"}
if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

PUBLIC_REPO_URL = "https://github.com/liuwen/qwen-asr-eval.git"  #@param {type:"string"}
PUBLIC_REPO_BRANCH = "main"  #@param {type:"string"}
REPO_DIR = Path("/content/qwen-asr-eval")  #@param {type:"string"}
DRIVE_ROOT = Path("/content/drive/MyDrive/asr")  #@param {type:"string"}
RUN_ID = "qwen3_asr_generic_run"  #@param {type:"string"}

AUDIO_ROOT = DRIVE_ROOT / "audio"
RAW_AUDIO_DIR = AUDIO_ROOT / "raw_inputs"
HF_CACHE_ROOT = DRIVE_ROOT / "hf_cache"
RUN_DIR = DRIVE_ROOT / "qwen-asr-eval" / "runs" / RUN_ID
OUT_DIR = RUN_DIR / "outputs"
WORK_DIR = Path("/content/asr-eval-work") / RUN_ID
CHUNK_DIR = WORK_DIR / "chunks"

for p in [AUDIO_ROOT, RAW_AUDIO_DIR, HF_CACHE_ROOT, OUT_DIR, WORK_DIR, CHUNK_DIR]:
    p.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_CACHE_ROOT)
os.environ["HF_HUB_CACHE"] = str(HF_CACHE_ROOT / "hub")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"

print("RUN_DIR:", RUN_DIR)
print("OUT_DIR:", OUT_DIR)
print("WORK_DIR:", WORK_DIR)
print("HF_HOME:", os.environ["HF_HOME"])


In [ ]:
#@title 1. Clone/update helper repo and install dependencies
import subprocess, sys, os

if REPO_DIR.exists():
    print("Repo exists; pulling latest")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", PUBLIC_REPO_BRANCH], check=False)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PUBLIC_REPO_BRANCH, PUBLIC_REPO_URL, str(REPO_DIR)], check=True)

src_path = str(REPO_DIR / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

subprocess.run(["apt-get", "-qq", "update"], check=True)
subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg", "jq"], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "qwen-asr", "huggingface_hub[hf_transfer]", "feedparser", "requests", "tqdm",
    "pandas", "numpy", "rapidfuzz", "soundfile", "librosa", "pydub", "jiwer",
    "opencc-python-reimplemented",
], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR), "--no-deps"], check=True)

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
    if HF_TOKEN:
        os.environ["HF_TOKEN"] = HF_TOKEN
        from huggingface_hub import login
        login(token=HF_TOKEN, add_to_git_credential=False)
        print("HF_TOKEN configured from Colab Secrets.")
    else:
        print("HF_TOKEN not set; continuing with public model download.")
except Exception as e:
    print("HF_TOKEN unavailable; continuing if public download works:", repr(e))

from asr_eval.presets import preset_table
import pandas as pd
print("Runtime presets:")
display(pd.DataFrame(preset_table()))


In [ ]:
#@title 2. Choose audio source
from pathlib import Path
import json
from asr_eval.sources import prepare_audio_source
from asr_eval.xiaoyuzhou import XIAOYUZHOU_EVAL_MANIFEST
from asr_eval.audio import ffprobe_duration, fmt_ts

SOURCE_TYPE = "drive_path"  #@param ["drive_path", "direct_url", "xiaoyuzhou"]
DRIVE_AUDIO_PATH = "/content/drive/MyDrive/asr/audio/raw_xiaoyuzhou/mandarin_long_sushi.audio"  #@param {type:"string"}
DIRECT_AUDIO_URL = ""  #@param {type:"string"}
XIAOYUZHOU_URL = "https://www.xiaoyuzhoufm.com/episode/69f08ec360313a2456c966c7"  #@param {type:"string"}
OUTPUT_FILENAME = ""  #@param {type:"string"}
FORCE_DOWNLOAD = False  #@param {type:"boolean"}

print("Available Xiaoyuzhou preset IDs:")
for item in XIAOYUZHOU_EVAL_MANIFEST:
    print(f"- {item['id']}: {item['scenario']} ({item['duration_min']} min) {item['episode_url']}")

source_info = prepare_audio_source(
    source_type=SOURCE_TYPE,
    drive_audio_path=DRIVE_AUDIO_PATH,
    direct_url=DIRECT_AUDIO_URL,
    xiaoyuzhou_url=XIAOYUZHOU_URL,
    raw_audio_dir=RAW_AUDIO_DIR,
    output_filename=OUTPUT_FILENAME or None,
    force_download=FORCE_DOWNLOAD,
)
AUDIO_PATH = Path(source_info["path"])
source_meta_path = OUT_DIR / "source_meta.json"
source_meta_path.write_text(json.dumps(source_info, ensure_ascii=False, indent=2), encoding="utf-8")

print("AUDIO_PATH:", AUDIO_PATH)
print("duration:", fmt_ts(ffprobe_duration(AUDIO_PATH)))
print("source meta saved:", source_meta_path)


In [ ]:
#@title 3. Choose runtime/language parameters
from asr_eval.presets import get_runtime_preset

RUNTIME_PRESET = "A100"  #@param ["T4", "A100", "H100", "CUSTOM"]
USE_CASE = "auto"  #@param ["auto", "zh", "en", "mixed", "cantonese"]
ASR_MODEL = "Qwen/Qwen3-ASR-1.7B"  #@param {type:"string"}

# CUSTOM values only apply when RUNTIME_PRESET == CUSTOM.
CUSTOM_CHUNK_SECONDS = 600  #@param {type:"integer"}
CUSTOM_OVERLAP_SECONDS = 5  #@param {type:"integer"}
CUSTOM_QWEN_BATCH_SIZE = 4  #@param {type:"integer"}
CUSTOM_MAX_INFERENCE_BATCH_SIZE = 4  #@param {type:"integer"}
CUSTOM_MAX_NEW_TOKENS = 4096  #@param {type:"integer"}

if RUNTIME_PRESET == "CUSTOM":
    CHUNK_SECONDS = int(CUSTOM_CHUNK_SECONDS)
    OVERLAP_SECONDS = int(CUSTOM_OVERLAP_SECONDS)
    QWEN_BATCH_SIZE = int(CUSTOM_QWEN_BATCH_SIZE)
    QWEN_MAX_INFERENCE_BATCH_SIZE = int(CUSTOM_MAX_INFERENCE_BATCH_SIZE)
    QWEN_MAX_NEW_TOKENS = int(CUSTOM_MAX_NEW_TOKENS)
else:
    preset = get_runtime_preset(RUNTIME_PRESET)
    CHUNK_SECONDS = preset.chunk_seconds
    OVERLAP_SECONDS = preset.overlap_seconds
    QWEN_BATCH_SIZE = preset.qwen_batch_size
    QWEN_MAX_INFERENCE_BATCH_SIZE = preset.qwen_max_inference_batch_size
    QWEN_MAX_NEW_TOKENS = preset.qwen_max_new_tokens

print(json.dumps({
    "runtime_preset": RUNTIME_PRESET,
    "use_case": USE_CASE,
    "chunk_seconds": CHUNK_SECONDS,
    "overlap_seconds": OVERLAP_SECONDS,
    "qwen_batch_size": QWEN_BATCH_SIZE,
    "qwen_max_inference_batch_size": QWEN_MAX_INFERENCE_BATCH_SIZE,
    "qwen_max_new_tokens": QWEN_MAX_NEW_TOKENS,
}, ensure_ascii=False, indent=2))


In [ ]:
#@title 4. Normalize to 16 kHz mono WAV and chunk
from asr_eval.audio import normalize_to_wav, chunk_wav, write_jsonl, ffprobe_duration, fmt_ts
import json

FORCE_REBUILD_CHUNKS = False  #@param {type:"boolean"}
NORMALIZED_WAV = WORK_DIR / "normalized_16k_mono.wav"
manifest_path = OUT_DIR / "chunks_manifest.jsonl"

if FORCE_REBUILD_CHUNKS or not NORMALIZED_WAV.exists():
    NORMALIZED_WAV = normalize_to_wav(AUDIO_PATH, NORMALIZED_WAV)
else:
    print("exists:", NORMALIZED_WAV)
print("normalized duration:", fmt_ts(ffprobe_duration(NORMALIZED_WAV)))

if FORCE_REBUILD_CHUNKS or not manifest_path.exists():
    chunks = chunk_wav(NORMALIZED_WAV, CHUNK_DIR, chunk_seconds=CHUNK_SECONDS, overlap_seconds=OVERLAP_SECONDS)
    write_jsonl(manifest_path, chunks)
else:
    chunks = []
    with manifest_path.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                chunks.append(json.loads(line))
print("chunks:", len(chunks))
print(json.dumps(chunks[:3], ensure_ascii=False, indent=2))


In [ ]:
#@title 5. Load Qwen3-ASR model
from asr_eval.qwen_runner import load_qwen_model, qwen_language_for_use_case

qwen_model = load_qwen_model(
    ASR_MODEL,
    max_inference_batch_size=QWEN_MAX_INFERENCE_BATCH_SIZE,
    max_new_tokens=QWEN_MAX_NEW_TOKENS,
    use_forced_aligner=False,
)
QWEN_LANGUAGE = qwen_language_for_use_case(USE_CASE)
print("Loaded:", ASR_MODEL)
print("QWEN_LANGUAGE:", QWEN_LANGUAGE)


In [ ]:
#@title 6. Transcribe with checkpoint/resume
import json, time
from asr_eval.qwen_runner import transcribe_chunks
from asr_eval.reporting import save_transcript
import pandas as pd

QWEN_JSONL = OUT_DIR / "qwen_chunks.jsonl"
STOP_AFTER_N_NEW_BATCHES = 0  #@param {type:"integer"}
# 0 means no artificial stop. Rerun this cell to resume after disconnect.

existing_rows = []
completed_ids = set()
if QWEN_JSONL.exists():
    with QWEN_JSONL.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                row = json.loads(line)
                existing_rows.append(row)
                completed_ids.add(int(row["chunk_id"]))
print("already completed chunks:", sorted(completed_ids))

pending_chunks = [c for c in chunks if int(c["chunk_id"]) not in completed_ids]
print("pending chunks:", len(pending_chunks), "of", len(chunks))
new_batches_done = 0

for i in range(0, len(pending_chunks), QWEN_BATCH_SIZE):
    batch = pending_chunks[i:i + QWEN_BATCH_SIZE]
    batch_ids = [int(c["chunk_id"]) for c in batch]
    print(f"\n=== batch {batch_ids}: {[(c['start_ts'], c['end_ts']) for c in batch]} ===")
    t0 = time.time()
    rows_new = transcribe_chunks(
        qwen_model,
        batch,
        language=QWEN_LANGUAGE,
        batch_size=QWEN_BATCH_SIZE,
        return_time_stamps=False,
        model_name=ASR_MODEL,
    )
    with QWEN_JSONL.open("a", encoding="utf-8") as f:
        for row in rows_new:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            f.flush()
    existing_rows.extend(rows_new)
    for row in rows_new:
        completed_ids.add(int(row["chunk_id"]))

    sorted_rows = sorted(existing_rows, key=lambda r: int(r["chunk_id"]))
    pd.DataFrame(sorted_rows).drop(columns=["time_stamps"], errors="ignore").to_csv(OUT_DIR / "qwen_chunks.csv", index=False)
    saved = save_transcript(OUT_DIR, "qwen", "Qwen3-ASR transcript", sorted_rows)
    print(f"saved batch {batch_ids}; elapsed {time.time()-t0:.1f}s; completed {len(completed_ids)}/{len(chunks)}")
    print(str(rows_new[0].get("text") or "")[:800])
    print("checkpoint transcript:", saved["md_path"])

    new_batches_done += 1
    if STOP_AFTER_N_NEW_BATCHES and new_batches_done >= STOP_AFTER_N_NEW_BATCHES:
        print("STOP_AFTER_N_NEW_BATCHES reached; rerun this cell to resume.")
        break

print("\ncompleted", len(completed_ids), "of", len(chunks))
print("jsonl:", QWEN_JSONL)
print("md:", OUT_DIR / "qwen_transcript_chunked.md")
print("txt:", OUT_DIR / "qwen_transcript_chunked.txt")


In [ ]:
#@title 7. Rebuild/save transcript artifacts from qwen_chunks.jsonl
import json
import pandas as pd
from asr_eval.reporting import save_transcript

QWEN_JSONL = OUT_DIR / "qwen_chunks.jsonl"
rows = []
with QWEN_JSONL.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))
rows = sorted(rows, key=lambda r: int(r["chunk_id"]))
pd.DataFrame(rows).drop(columns=["time_stamps"], errors="ignore").to_csv(OUT_DIR / "qwen_chunks.csv", index=False)
saved = save_transcript(OUT_DIR, "qwen", "Qwen3-ASR transcript", rows)
print(json.dumps({
    "rows": len(rows),
    "text_chars": sum(len(str(r.get("text") or "")) for r in rows),
    "first_range": [rows[0].get("start_ts"), rows[0].get("end_ts")],
    "last_range": [rows[-1].get("start_ts"), rows[-1].get("end_ts")],
    "csv": str(OUT_DIR / "qwen_chunks.csv"),
    "md": str(saved["md_path"]),
    "txt": str(saved["txt_path"]),
}, ensure_ascii=False, indent=2))


In [ ]:
#@title 8. Colab GenAI text-only quality triage
RUN_COLAB_AI_TEXT_JUDGE = True  #@param {type:"boolean"}
COLAB_AI_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
MAX_TEXT_JUDGE_CHUNKS = 6  #@param {type:"integer"}

if RUN_COLAB_AI_TEXT_JUDGE:
    from asr_eval.colab_ai_judge import judge_chunks_colab_ai, select_eval_chunk_ids
    eval_chunk_ids = select_eval_chunk_ids(rows, whisper_rows=None, max_chunks=MAX_TEXT_JUDGE_CHUNKS)
    print("Selected chunk IDs:", eval_chunk_ids)
    reports = judge_chunks_colab_ai(rows, whisper_rows=None, use_case=USE_CASE, model_name=COLAB_AI_MODEL, max_chunks=MAX_TEXT_JUDGE_CHUNKS)
    judge_path = OUT_DIR / "colab_ai_text_judge.json"
    judge_path.write_text(json.dumps(reports, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", judge_path)
    import pandas as pd
    display(pd.DataFrame(reports))
else:
    reports = []
    print("Skipped.")


In [ ]:
#@title 9A. Build proofreading context guide
import json, re
from google.colab import ai

PROOFREAD_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
PROOFREAD_PROMPT_LANGUAGE = "zh_cn"  #@param ["zh_cn", "en"]
MAX_CONTEXT_TRANSCRIPT_CHARS = 90000  #@param {type:"integer"}

full_transcript = "\n\n".join(f"[{r.get('start_ts')} - {r.get('end_ts')}]\n{str(r.get('text') or '')}" for r in rows)
context_text = full_transcript[:MAX_CONTEXT_TRANSCRIPT_CHARS]

if PROOFREAD_PROMPT_LANGUAGE == "en":
    prompt = f"""
You are the high-fidelity proofreading director for an ASR transcript. You only see text; you cannot hear the audio.
Build a context guide for later chunk-by-chunk proofreading.
Principles: do not claim audio verification; do not expand; do not change opinions; only organize theme, structure, proper nouns, terms, quote candidates, and recurring homophone/ASR-risk patterns. Mark uncertain items as uncertain.
Return JSON only:
{{"judge_type":"text_only_proofread_context","audio_faithfulness_verified":false,"main_theme":"...","structure_outline":[{{"section":"...","time_hint":"...","summary":"..."}}],"key_entities":[{{"name":"...","type":"person|place|book|poem|term|other","confidence":"high|medium|low","note":"..."}}],"likely_quote_or_classical_text_candidates":[{{"text":"...","confidence":"high|medium|low","note":"..."}}],"recurring_asr_risk_patterns":[{{"pattern":"...","likely_correction":"...","confidence":"high|medium|low","note":"..."}}],"proofreading_rules":["..."]}}

Transcript:
{context_text}
""".strip()
else:
    prompt = f"""
你是 ASR 转写稿的高保真校对总监。你只能看到文字，不能听音频。
建立一个后续逐段校对使用的上下文指南。
原则：不声称验证音频；不扩写；不改变观点；只整理主题、结构、专名、术语、引文候选、常见同音误识别风险。不确定则标 uncertain。
只返回 JSON：
{{"judge_type":"text_only_proofread_context","audio_faithfulness_verified":false,"main_theme":"...","structure_outline":[{{"section":"...","time_hint":"...","summary":"..."}}],"key_entities":[{{"name":"...","type":"person|place|book|poem|term|other","confidence":"high|medium|low","note":"..."}}],"likely_quote_or_classical_text_candidates":[{{"text":"...","confidence":"high|medium|low","note":"..."}}],"recurring_asr_risk_patterns":[{{"pattern":"...","likely_correction":"...","confidence":"high|medium|low","note":"..."}}],"proofreading_rules":["..."]}}

转写稿：
{context_text}
""".strip()

raw = ai.generate_text(prompt, model_name=PROOFREAD_MODEL)
raw_text = str(raw)
try:
    context_guide = json.loads(raw_text)
except Exception:
    m = re.search(r"\{.*\}", raw_text, flags=re.S)
    context_guide = json.loads(m.group(0)) if m else {"raw_text": raw_text}
context_guide["_colab_ai_model"] = PROOFREAD_MODEL
context_guide["_prompt_language"] = PROOFREAD_PROMPT_LANGUAGE
context_guide["_source_chars_sent"] = len(context_text)
context_path = OUT_DIR / "qwen_proofread_context.json"
context_path.write_text(json.dumps(context_guide, ensure_ascii=False, indent=2), encoding="utf-8")
print("Saved:", context_path)
print(json.dumps(context_guide, ensure_ascii=False, indent=2)[:6000])


In [ ]:
#@title 9B. High-fidelity proofread chunks with checkpoint/resume
import json, re, time
from google.colab import ai

PROOFREAD_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
PROOFREAD_PROMPT_LANGUAGE = "zh_cn"  #@param ["zh_cn", "en"]
REPROOFREAD_EXISTING = False  #@param {type:"boolean"}
STOP_AFTER_N_PROOFREAD_CHUNKS = 0  #@param {type:"integer"}

context_path = OUT_DIR / "qwen_proofread_context.json"
if context_path.exists():
    context_guide = json.loads(context_path.read_text(encoding="utf-8"))

proofread_jsonl = OUT_DIR / "qwen_proofread_chunks.jsonl"
completed = {}
if proofread_jsonl.exists() and not REPROOFREAD_EXISTING:
    with proofread_jsonl.open("r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                obj = json.loads(line)
                completed[int(obj["chunk_id"])] = obj
print("already proofread chunks:", sorted(completed))

row_by_id = {int(r["chunk_id"]): r for r in rows}
all_ids = sorted(row_by_id)

def parse_json_object(raw_text: str):
    try:
        return json.loads(raw_text)
    except Exception:
        m = re.search(r"\{.*\}", raw_text, flags=re.S)
        if m:
            try:
                return json.loads(m.group(0))
            except Exception:
                pass
    return None

def excerpt(cid, limit=900):
    if cid not in row_by_id:
        return ""
    r = row_by_id[cid]
    return f"[{r.get('start_ts')} - {r.get('end_ts')}]\n{str(r.get('text') or '')[:limit]}"

new_done = 0
for cid in all_ids:
    if cid in completed and not REPROOFREAD_EXISTING:
        continue
    r = row_by_id[cid]
    original = str(r.get("text") or "")
    if PROOFREAD_PROMPT_LANGUAGE == "en":
        prompt = f"""
You are a high-fidelity proofreader for an ASR transcript. You only see text; you cannot hear the audio.
Principles: preserve the original expression 100%; do not expand, delete ideas, or change order; preserve spoken style; only make conservative corrections: punctuation, sentence/paragraph boundaries, obvious homophones, proper nouns, and quote errors. If uncertain, keep the original and record uncertainty.
Context guide: {json.dumps(context_guide, ensure_ascii=False)[:20000]}
Previous excerpt: {excerpt(cid-1)}
Current timestamp: [{r.get('start_ts')} - {r.get('end_ts')}]
Current original text: {original}
Next excerpt: {excerpt(cid+1)}
Return JSON only: {{"chunk_id":{cid},"start_ts":"{r.get('start_ts')}","end_ts":"{r.get('end_ts')}","audio_faithfulness_verified":false,"proofread_text":"complete proofread text","section_heading":"optional heading","correction_log":[{{"original":"...","corrected":"...","reason":"punctuation|homophone|proper_noun|quote|sectioning|other","confidence":"high|medium|low"}}],"uncertain_items":["..."]}}
""".strip()
    else:
        prompt = f"""
你是 ASR 转写稿的高保真校对员。你只能看到文字，不能听音频。
原则：100% 忠于原表达；不扩写、不删观点、不改变顺序；保留口语风格；只做保守校对：标点、断句、段落、明显同音字/专名/引文错误；不确定就保留原文并记录 uncertainty。
上下文指南：{json.dumps(context_guide, ensure_ascii=False)[:20000]}
上一段片段：{excerpt(cid-1)}
当前时间戳：[{r.get('start_ts')} - {r.get('end_ts')}]
当前原文：{original}
下一段片段：{excerpt(cid+1)}
只返回 JSON：{{"chunk_id":{cid},"start_ts":"{r.get('start_ts')}","end_ts":"{r.get('end_ts')}","audio_faithfulness_verified":false,"proofread_text":"校对后的完整文本","section_heading":"可选小标题","correction_log":[{{"original":"...","corrected":"...","reason":"punctuation|homophone|proper_noun|quote|sectioning|other","confidence":"high|medium|low"}}],"uncertain_items":["..."]}}
""".strip()
    print(f"\n=== proofread chunk {cid}/{all_ids[-1]} {r.get('start_ts')} - {r.get('end_ts')} ===")
    t0 = time.time()
    raw = ai.generate_text(prompt, model_name=PROOFREAD_MODEL)
    data = parse_json_object(str(raw))
    if data is None or "proofread_text" not in data:
        data = {"chunk_id": cid, "start_ts": r.get("start_ts"), "end_ts": r.get("end_ts"), "audio_faithfulness_verified": False, "proofread_text": original, "section_heading": "", "correction_log": [], "uncertain_items": ["Model response could not be parsed; kept original text."], "raw_response": str(raw)}
    data.setdefault("chunk_id", cid)
    data.setdefault("audio_faithfulness_verified", False)
    data["original_text"] = original
    data["_colab_ai_model"] = PROOFREAD_MODEL
    data["_prompt_language"] = PROOFREAD_PROMPT_LANGUAGE
    data["_elapsed_sec"] = round(time.time() - t0, 2)
    completed[cid] = data
    with proofread_jsonl.open("w", encoding="utf-8") as f:
        for k in sorted(completed):
            f.write(json.dumps(completed[k], ensure_ascii=False) + "\n")
    print("elapsed:", data["_elapsed_sec"], "corrections:", len(data.get("correction_log", [])))
    print(str(data.get("proofread_text") or "")[:800])
    new_done += 1
    if STOP_AFTER_N_PROOFREAD_CHUNKS and new_done >= STOP_AFTER_N_PROOFREAD_CHUNKS:
        print("STOP_AFTER_N_PROOFREAD_CHUNKS reached; rerun to resume.")
        break
print("proofread chunks:", len(completed), "of", len(all_ids))
print("jsonl:", proofread_jsonl)


In [ ]:
#@title 9C. Assemble proofread transcript and correction log
import json
proofread_jsonl = OUT_DIR / "qwen_proofread_chunks.jsonl"
proofread_rows = []
with proofread_jsonl.open("r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            proofread_rows.append(json.loads(line))
proofread_rows = sorted(proofread_rows, key=lambda r: int(r["chunk_id"]))
md_lines = ["# Qwen3-ASR transcript — high-fidelity proofread\n\n"]
txt_lines, corrections = [], []
for r in proofread_rows:
    start, end = r.get("start_ts"), r.get("end_ts")
    heading = (r.get("section_heading") or "").strip()
    text = (r.get("proofread_text") or "").strip()
    md_lines.append(f"## [{start} - {end}]" + (f" {heading}" if heading else "") + "\n\n" + text + "\n\n")
    txt_lines.extend([f"[{start} - {end}]" + (f" {heading}" if heading else ""), text, ""])
    for item in r.get("correction_log", []) or []:
        corrections.append({"chunk_id": r.get("chunk_id"), "start_ts": start, "end_ts": end, **item})
md_path = OUT_DIR / "qwen_proofread_transcript_chunked.md"
txt_path = OUT_DIR / "qwen_proofread_transcript_chunked.txt"
corrections_path = OUT_DIR / "qwen_proofread_corrections.json"
md_path.write_text("".join(md_lines), encoding="utf-8")
txt_path.write_text("\n".join(txt_lines), encoding="utf-8")
corrections_path.write_text(json.dumps(corrections, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps({"chunks": len(proofread_rows), "proofread_chars": sum(len(str(r.get("proofread_text") or "")) for r in proofread_rows), "corrections": len(corrections), "md": str(md_path), "txt": str(txt_path), "corrections_json": str(corrections_path)}, ensure_ascii=False, indent=2))


In [ ]:
#@title 10. Optional enhancements: highlights, translation, mindmap, research leads
from google.colab import ai
import json, re

RUN_ENHANCEMENTS = False  #@param {type:"boolean"}
ENHANCEMENT_MODEL = "google/gemini-3.5-flash"  #@param {type:"string"}
TRANSLATE_TO_ENGLISH = False  #@param {type:"boolean"}
MAX_ENHANCEMENT_CHARS = 120000  #@param {type:"integer"}

if RUN_ENHANCEMENTS:
    transcript_path = OUT_DIR / ("qwen_proofread_transcript_chunked.txt" if (OUT_DIR / "qwen_proofread_transcript_chunked.txt").exists() else "qwen_transcript_chunked.txt")
    transcript = transcript_path.read_text(encoding="utf-8")[:MAX_ENHANCEMENT_CHARS]
    prompt = f"""
你是长音频转写稿的研究助理。你只能基于文字，不要声称听过音频。
请产出 JSON：
{{"highlights":["..."],"theme_outline":[{{"title":"...","points":["..."]}}],"mindmap_markdown":"...","domain_research_leads":[{{"topic":"...","why":"...","search_queries":["..."]}}],"translation_note":"...","english_summary":"..."}}
如果 TRANSLATE_TO_ENGLISH={TRANSLATE_TO_ENGLISH}，请提供更详细 english_summary；否则简短即可。

Transcript:
{transcript}
""".strip()
    raw = ai.generate_text(prompt, model_name=ENHANCEMENT_MODEL)
    raw_text = str(raw)
    try:
        enhancements = json.loads(raw_text)
    except Exception:
        m = re.search(r"\{.*\}", raw_text, flags=re.S)
        enhancements = json.loads(m.group(0)) if m else {"raw_text": raw_text}
    path = OUT_DIR / "transcript_enhancements.json"
    path.write_text(json.dumps(enhancements, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved:", path)
    print(json.dumps(enhancements, ensure_ascii=False, indent=2)[:8000])
else:
    print("Skipped enhancements. Set RUN_ENHANCEMENTS=True to run.")
